# PatternAnomalyDetector v1 검증 (ANOM-002)

**목적**: `pattern.py` 탐지 로직 2가지 검증
1. Isolation Forest — 이상 스코어 분포, 이상 구간 시각화, contamination 민감도
2. Periodicity detection — 시간대별 패턴 편차, std_multiplier 민감도

**데이터**: GCS house_049 (2023-10-12 ~ 2023-11-11)

---

## 탐지 방법론

### 전체 구조

```
가전별 1분 시계열 (NILM 출력)
    │
    ├─ [Isolation Forest]  fit(베이스라인) → predict(평가 구간)
    │      피처: [power_w, is_on, hour, dayofweek]
    │      출력: 이상 비율 → severity 판정
    │
    └─ [Periodicity]  베이스라인 시간대별 평균 프로파일 vs 평가 구간 프로파일
           기준: |편차| > baseline_std × std_multiplier
           출력: PERIODICITY_CHANGE (LOW)
```

### ANOM-001 vs ANOM-002 역할 분리

| | ANOM-001 (statistical) | ANOM-002 (pattern) |
|-|------------------------|--------------------|
| 방식 | 규칙 기반 (임계값 비교) | 학습 기반 (IF) + 규칙 (periodicity) |
| 탐지 대상 | 소비량·피크·런타임 **증가** | 비정상 패턴 조합, 시간대 편차 |
| 한계 | 감소 이상·시간대 이상 미탐지 | 베이스라인 데이터 부족 시 과탐 |
| 보완 관계 | IF가 잡는 복합 패턴을 ANOM-001은 못 잡음 | 소비량 절대값 증가는 ANOM-001이 더 직접적 |

## 0. 환경 설정

In [ ]:
!pip install -q gcsfs pyarrow pandas numpy matplotlib seaborn scikit-learn
!apt-get -qq install fonts-nanum
!rm -rf ~/.cache/matplotlib
print('설치 완료')

In [ ]:
from google.colab import auth
auth.authenticate_user()
print('GCP 인증 완료')

In [ ]:
import gcsfs, os, sys
gcs = gcsfs.GCSFileSystem()

REPO_DIR = '/content/ax_nilm'
if not os.path.exists(REPO_DIR):
    !git clone https://github.com/dhwang0803-glitch/ax_nilm {REPO_DIR} -q
    print('클론 완료')
else:
    !git -C {REPO_DIR} pull -q && echo '최신화 완료'

for p in [f'{REPO_DIR}/nilm-engine/src', f'{REPO_DIR}/anomaly-detection']:
    if p not in sys.path:
        sys.path.insert(0, p)
print('경로 설정 완료')

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import gc
from datetime import timedelta, datetime
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import seaborn as sns
from sklearn.ensemble import IsolationForest

from acquisition.gcs_loader import (
    list_channels_gcs, load_channel_data_gcs,
    get_appliance_name_gcs, get_house_start_date_gcs,
)

sns.set_style('whitegrid')
NANUM_PATH = '/usr/share/fonts/truetype/nanum/NanumGothic.ttf'
fm.fontManager.addfont(NANUM_PATH)
plt.rcParams['font.family'] = 'NanumGothic'
plt.rcParams['axes.unicode_minus'] = False
print('imports OK  |  폰트:', plt.rcParams['font.family'])

In [ ]:
HOUSE         = 'house_049'
BUCKET_PREFIX = 'ax-nilm-data-dhwang0803-us/nilm/training_dev10'
LABEL_PATH    = 'ax-nilm-data-dhwang0803-us/nilm/labels/training.parquet'
RESAMPLE_FREQ = '1min'
CHUNK_DAYS    = 7

TARGET_APPLIANCES = {'세탁기', '전자레인지', '전기밥솥', '에어컨', '일반 냉장고'}

ON_THRESHOLDS = {
    '세탁기': 10.0, '전자레인지': 10.0, '전기밥솥': 5.0,
    '에어컨':  2.0, '일반 냉장고':  5.0,
}
DEFAULT_ON_THR = 10.0

# ANOM-002 하이퍼파라미터 (pattern.py 기본값)
CONTAMINATION         = 0.05   # IF: 정상 데이터 중 이상 비율 가정
N_ESTIMATORS          = 100
PERIODICITY_STD_MULT  = 2.0    # periodicity: 시간대 편차 허용 배수

print('설정 완료')

## 1. GCS 데이터 로딩

In [ ]:
AGGREGATE_KW = {'메인', '분전반', '집계', 'main', 'aggregate'}
POWER_COL = None
appliance_series = {}

channels = list_channels_gcs(gcs, HOUSE, bucket_prefix=BUCKET_PREFIX)
ch_map = {}
for ch in channels:
    name = get_appliance_name_gcs(gcs, HOUSE, ch, label_path=LABEL_PATH)
    ch_map[ch] = str(name) if (name and not pd.isna(name)) else ch

for ch, appliance in ch_map.items():
    if any(kw in appliance for kw in AGGREGATE_KW):
        continue
    if TARGET_APPLIANCES and appliance not in TARGET_APPLIANCES:
        continue
    try:
        start_date = get_house_start_date_gcs(gcs, HOUSE, ch, bucket_prefix=BUCKET_PREFIX)
    except FileNotFoundError:
        continue

    end_date = start_date + timedelta(days=30)
    chunks, current = [], start_date
    while current <= end_date:
        chunk_end = min(current + timedelta(days=CHUNK_DAYS - 1), end_date)
        try:
            raw = load_channel_data_gcs(gcs, HOUSE, ch,
                                        date_range=(str(current), str(chunk_end)),
                                        bucket_prefix=BUCKET_PREFIX)
            raw = raw.set_index('date_time').sort_index()
            if POWER_COL is None:
                for c in ['agg_power', 'power_w', 'power', 'active_power', 'value']:
                    if c in raw.columns:
                        POWER_COL = c; break
                if POWER_COL is None:
                    POWER_COL = raw.select_dtypes(include='number').columns[0]
                print(f'전력 컬럼: {POWER_COL}')
            chunks.append(raw[POWER_COL].resample(RESAMPLE_FREQ).mean())
            del raw; gc.collect()
        except Exception as e:
            print(f'{ch} {current}~{chunk_end} 오류: {e}')
        current = chunk_end + timedelta(days=1)

    if chunks:
        s = pd.concat(chunks)
        appliance_series[appliance] = s
        print(f'  {appliance} ({ch}): {len(s):,}행  {s.index.min().date()} ~ {s.index.max().date()}')
    del chunks; gc.collect()

wide_1min = pd.DataFrame(appliance_series)

# 베이스라인 / 평가 구간 분리 (24h eval)
end_ts       = wide_1min.index.max()
eval_start   = end_ts - timedelta(hours=24)
base_start   = end_ts - timedelta(weeks=4)
baseline_1min = wide_1min[(wide_1min.index >= base_start) & (wide_1min.index < eval_start)]
eval_1min     = wide_1min[(wide_1min.index >= eval_start)]

print(f'\n베이스라인: {baseline_1min.index.min().date()} ~ {baseline_1min.index.max().date()}')
print(f'평가 구간:  {eval_1min.index.min().date()} ~ {eval_1min.index.max().date()}')
print(f'shape: {wide_1min.shape}')

## 2. 피처 분포 확인

`pattern.py`의 `_features()` = `[power_w, is_on, hour, dayofweek]`

IF가 학습할 피처가 가전별 패턴을 충분히 구분하는지 시각화.

In [ ]:
for name in wide_1min.columns:
    on_thr = ON_THRESHOLDS.get(name, DEFAULT_ON_THR)
    df = baseline_1min[[name]].dropna().copy()
    df.columns = ['power_w']
    df['is_on']      = (df['power_w'] >= on_thr).astype(int)
    df['hour']       = df.index.hour
    df['dayofweek']  = df.index.dayofweek

    fig, axes = plt.subplots(1, 4, figsize=(16, 3))

    axes[0].hist(df['power_w'], bins=50, color='steelblue', edgecolor='white')
    axes[0].set_title('power_w 분포')
    axes[0].set_xlabel('W')

    axes[1].bar([0, 1], df['is_on'].value_counts().sort_index(),
                color=['lightgray', 'tomato'], edgecolor='white')
    axes[1].set_xticks([0, 1]); axes[1].set_xticklabels(['OFF', 'ON'])
    axes[1].set_title(f'is_on 비율  (임계 {on_thr}W)')

    hourly_mean = df.groupby('hour')['power_w'].mean()
    axes[2].bar(hourly_mean.index, hourly_mean.values, color='steelblue', edgecolor='white')
    axes[2].set_title('시간대별 평균 (W)')
    axes[2].set_xlabel('hour')

    daily_mean = df.groupby('dayofweek')['power_w'].mean()
    axes[3].bar(daily_mean.index, daily_mean.values, color='steelblue', edgecolor='white')
    axes[3].set_xticks(range(7))
    axes[3].set_xticklabels(['월','화','수','목','금','토','일'])
    axes[3].set_title('요일별 평균 (W)')

    plt.suptitle(f'{name} — 피처 분포 (베이스라인)', fontsize=10)
    plt.tight_layout()
    plt.show()
    print(f'  {name}: power_w 평균={df["power_w"].mean():.1f}W  ON비율={df["is_on"].mean()*100:.1f}%')

## 3. Isolation Forest 학습 & 이상 스코어 분포

In [ ]:
def make_features(series: pd.Series, on_thr: float) -> np.ndarray:
    df = series.dropna().to_frame('power_w')
    df['is_on']     = (df['power_w'] >= on_thr).astype(int)
    df['hour']      = df.index.hour
    df['dayofweek'] = df.index.dayofweek
    return df[['power_w', 'is_on', 'hour', 'dayofweek']].values

models = {}
for name in wide_1min.columns:
    on_thr = ON_THRESHOLDS.get(name, DEFAULT_ON_THR)
    feats  = make_features(baseline_1min[name], on_thr)
    if len(feats) < 10:
        continue
    m = IsolationForest(contamination=CONTAMINATION,
                        n_estimators=N_ESTIMATORS,
                        random_state=42)
    m.fit(feats)
    models[name] = m
    print(f'  {name}: 학습 완료  ({len(feats):,}샘플)')

print(f'\n총 {len(models)}개 가전 모델 학습 완료')

In [ ]:
# 베이스라인 스코어 분포 vs 평가 구간 스코어 분포
for name, model in models.items():
    on_thr = ON_THRESHOLDS.get(name, DEFAULT_ON_THR)

    bl_feats = make_features(baseline_1min[name], on_thr)
    ev_feats = make_features(eval_1min[name], on_thr)

    bl_scores = model.decision_function(bl_feats)
    ev_scores = model.decision_function(ev_feats)
    ev_preds  = model.predict(ev_feats)
    anomaly_ratio = (ev_preds == -1).mean()

    fig, axes = plt.subplots(1, 2, figsize=(12, 3))

    axes[0].hist(bl_scores, bins=50, alpha=0.6, color='steelblue', label='베이스라인')
    axes[0].hist(ev_scores, bins=50, alpha=0.6, color='tomato',    label='평가 구간')
    axes[0].axvline(0, color='black', linestyle='--', linewidth=1, label='이상 경계 (score=0)')
    axes[0].set_title('Anomaly Score 분포')
    axes[0].set_xlabel('score  (음수 = 이상)')
    axes[0].legend(fontsize=8)

    bl_ratio = (model.predict(bl_feats) == -1).mean()
    axes[1].bar(['베이스라인', '평가 구간'], [bl_ratio * 100, anomaly_ratio * 100],
                color=['steelblue', 'tomato'], edgecolor='white')
    axes[1].axhline(CONTAMINATION * 100, color='orange', linestyle='--',
                    label=f'contamination={CONTAMINATION*100:.0f}%')
    axes[1].set_title('이상 판정 비율 (%)')
    axes[1].set_ylabel('%')
    axes[1].legend(fontsize=8)

    severity = 'HIGH' if anomaly_ratio > 0.3 else 'MEDIUM' if anomaly_ratio > 0.1 else 'LOW' if anomaly_ratio > 0 else 'NORMAL'
    plt.suptitle(f'{name}  |  평가 구간 이상 비율 {anomaly_ratio*100:.1f}%  [{severity}]', fontsize=10)
    plt.tight_layout()
    plt.show()

### 3번 해석 — 이상 스코어 분포

- **score < 0**: Isolation Forest가 이상으로 판정한 구간
- **베이스라인 이상 비율** ≈ `contamination` 설정값(5%)이 정상 — 학습 데이터에서 5%는 이상으로 가정
- **평가 구간 이상 비율 > contamination**: 베이스라인 대비 패턴 변화 있음
- **평가 구간 이상 비율 ≈ contamination**: 정상 범위 — 알림 불필요

> 24h 평가 구간은 데이터 포인트가 1,440개(1분 × 24h)로 적어 분포가 불안정할 수 있음.
> 이상 비율 임계값: LOW > 0%, MEDIUM > 10%, HIGH > 30%

## 4. 이상 구간 시각화

In [ ]:
for name, model in models.items():
    on_thr   = ON_THRESHOLDS.get(name, DEFAULT_ON_THR)
    ev_series = eval_1min[name].dropna()
    if ev_series.empty:
        continue

    ev_feats = make_features(ev_series, on_thr)
    preds    = model.predict(ev_feats)
    scores   = model.decision_function(ev_feats)

    fig, axes = plt.subplots(2, 1, figsize=(14, 5), sharex=True)

    axes[0].plot(ev_series.index, ev_series.values,
                 color='steelblue', linewidth=0.8, label='power_w')
    anom_idx = ev_series.index[preds == -1]
    axes[0].scatter(anom_idx, ev_series.values[preds == -1],
                    color='red', s=20, zorder=5, label=f'이상 ({len(anom_idx)}개)')
    axes[0].axhline(on_thr, color='gray', linestyle='--', linewidth=0.8,
                    label=f'ON 임계 {on_thr}W')
    axes[0].set_title(f'{name} — 이상 구간 마커')
    axes[0].set_ylabel('W')
    axes[0].legend(fontsize=8)

    axes[1].plot(ev_series.index, scores, color='gray', linewidth=0.8)
    axes[1].fill_between(ev_series.index, scores, 0,
                         where=(scores < 0), color='tomato', alpha=0.4, label='이상 구간')
    axes[1].axhline(0, color='red', linestyle='--', linewidth=1)
    axes[1].set_title('Anomaly Score  (0 미만 = 이상)')
    axes[1].set_ylabel('score')
    axes[1].legend(fontsize=8)

    plt.tight_layout()
    plt.show()

### 4번 해석 — 이상 구간 시각화

확인 포인트:
- 이상 마커(빨간 점)가 **실제 비정상 구간**(급격한 전력 변화, 비정상 시간대)에 집중되는가?
- 이상 마커가 **전체에 균일하게 분산**된다면 → contamination이 너무 높거나, 피처가 부족
- Anomaly Score가 낮은 구간(음수 깊음)이 시각적으로 이상해 보이는 구간과 일치하는가?

> 24h 평가 구간이 짧아 이상 구간이 눈에 잘 안 보일 수 있음 — 섹션 6에서 더 긴 윈도우로 확인.

## 5. 시간대별 패턴 비교 (Periodicity Detection)

In [ ]:
for name in wide_1min.columns:
    bl = baseline_1min[name].dropna()
    ev = eval_1min[name].dropna()
    if bl.empty or ev.empty:
        continue

    bl_hourly = bl.groupby(bl.index.hour).mean()
    ev_hourly = ev.groupby(ev.index.hour).mean()
    bl_std    = bl.groupby(bl.index.hour).std()

    common = bl_hourly.index.intersection(ev_hourly.index)
    diff   = (ev_hourly[common] - bl_hourly[common]).abs()
    thr    = bl_std[common].mean() * PERIODICITY_STD_MULT
    triggered = diff.max() > thr

    fig, axes = plt.subplots(1, 2, figsize=(12, 3))

    axes[0].plot(bl_hourly.index, bl_hourly.values, color='steelblue', label='베이스라인')
    axes[0].fill_between(bl_hourly.index,
                         (bl_hourly - bl_std * PERIODICITY_STD_MULT).clip(lower=0),
                         bl_hourly + bl_std * PERIODICITY_STD_MULT,
                         alpha=0.15, color='steelblue', label=f'±{PERIODICITY_STD_MULT}σ')
    axes[0].plot(ev_hourly.index, ev_hourly.values,
                 color='tomato', linestyle='--', label='평가 구간')
    axes[0].set_title(f'{name} 시간대별 패턴')
    axes[0].set_xlabel('hour'); axes[0].set_ylabel('W')
    axes[0].legend(fontsize=8)

    axes[1].bar(common, diff[common], color='tomato', edgecolor='white', label='|편차|')
    axes[1].axhline(thr, color='orange', linestyle='--',
                    label=f'임계 ({PERIODICITY_STD_MULT}σ = {thr:.1f}W)')
    axes[1].set_title('시간대별 |편차|')
    axes[1].set_xlabel('hour'); axes[1].set_ylabel('W')
    axes[1].legend(fontsize=8)

    status = '⚠️ PERIODICITY_CHANGE 탐지' if triggered else '✅ 정상 범위'
    plt.suptitle(f'{name}  |  max|편차|={diff.max():.1f}W  임계={thr:.1f}W  {status}', fontsize=9)
    plt.tight_layout()
    plt.show()

### 5번 해석 — 시간대별 패턴 비교

- **음영(±Nσ)** 안에 평가 구간 프로파일이 들어오면 정상
- **편차 막대**가 임계선을 넘는 시간대가 있으면 `PERIODICITY_CHANGE (LOW)` 발생
- 24h 평가 구간이 하루치라 시간대별 대표값이 불안정할 수 있음 (점 1개/시간)

| 가전 | 기대 패턴 | 탐지 의미 |
|------|-----------|----------|
| **세탁기** | 낮 시간대 집중 | 새벽 가동 → 생활 패턴 변화 |
| **전기밥솥** | 아침·저녁 피크 | 식사 시간 변화 |
| **에어컨** | 낮~저녁 집중 | 야간 가동 증가 → 과다 사용 |
| **일반 냉장고** | 24h 평탄 | 특정 시간대 급증 → 문 열림/성능 저하 |

## 6. Hyperparameter Sensitivity

### 6-A. contamination (IF)

In [ ]:
contaminations = [0.01, 0.03, 0.05, 0.10, 0.15]

rows = []
for name in wide_1min.columns:
    on_thr   = ON_THRESHOLDS.get(name, DEFAULT_ON_THR)
    bl_feats = make_features(baseline_1min[name], on_thr)
    ev_feats = make_features(eval_1min[name], on_thr)
    if len(bl_feats) < 10:
        continue
    for c in contaminations:
        m = IsolationForest(contamination=c, n_estimators=N_ESTIMATORS, random_state=42)
        m.fit(bl_feats)
        ev_ratio = (m.predict(ev_feats) == -1).mean()
        severity = 'HIGH' if ev_ratio > 0.3 else 'MEDIUM' if ev_ratio > 0.1 else 'LOW' if ev_ratio > 0 else '-'
        rows.append({'가전': name, 'contamination': c,
                     '평가이상비율': round(ev_ratio * 100, 1), 'severity': severity})

sens_df = pd.DataFrame(rows)

fig, axes = plt.subplots(1, len(wide_1min.columns), figsize=(4 * len(wide_1min.columns), 4))
if len(wide_1min.columns) == 1:
    axes = [axes]
for ax, name in zip(axes, wide_1min.columns):
    sub = sens_df[sens_df['가전'] == name]
    ax.plot(sub['contamination'] * 100, sub['평가이상비율'],
            marker='o', color='steelblue')
    ax.axhline(10, color='orange', linestyle=':', linewidth=0.8, label='MEDIUM 기준 10%')
    ax.axhline(30, color='red',    linestyle=':', linewidth=0.8, label='HIGH 기준 30%')
    ax.set_title(name, fontsize=9)
    ax.set_xlabel('contamination (%)')
    ax.set_ylabel('평가 이상 비율 (%)')
    ax.legend(fontsize=7)

plt.suptitle('contamination 민감도 — 평가 구간 이상 비율 변화', fontsize=10)
plt.tight_layout()
plt.show()
print(sens_df.pivot(index='contamination', columns='가전', values='평가이상비율').to_string())

### 6-A 해석 — contamination 민감도

- contamination이 낮을수록 IF의 이상 판정 기준이 엄격해짐 → 평가 이상 비율 감소
- **contamination=0.05** 기준 가전별 평가 이상 비율이 MEDIUM(10%) 이하면 적절
- 특정 가전에서 contamination과 무관하게 이상 비율이 높으면 → 피처 부족 또는 실제 이상 가능성

> 권장: 평가 이상 비율이 contamination 설정값 ±2%p 범위에서 안정적이면 현행 유지.

### 6-B. periodicity_std_multiplier

In [ ]:
multipliers = [1.0, 1.5, 2.0, 2.5, 3.0]

rows2 = []
for name in wide_1min.columns:
    bl = baseline_1min[name].dropna()
    ev = eval_1min[name].dropna()
    if bl.empty or ev.empty:
        continue
    bl_h   = bl.groupby(bl.index.hour).mean()
    ev_h   = ev.groupby(ev.index.hour).mean()
    bl_std = bl.groupby(bl.index.hour).std()
    common = bl_h.index.intersection(ev_h.index)
    if len(common) < 6:
        continue
    diff   = (ev_h[common] - bl_h[common]).abs()
    for mult in multipliers:
        thr     = bl_std[common].mean() * mult
        trigger = diff.max() > thr
        rows2.append({'가전': name, 'std_mult': mult,
                      'max_diff': round(diff.max(), 1),
                      'threshold': round(thr, 1),
                      '탐지': '✅' if trigger else '-'})

sens2_df = pd.DataFrame(rows2)
print(sens2_df.pivot(index='std_mult', columns='가전', values='탐지').to_string())
print()
print('max|편차| vs threshold:')
print(sens2_df[sens2_df['std_mult'] == 2.0][['가전', 'max_diff', 'threshold', '탐지']].to_string(index=False))

### 6-B 해석 — periodicity_std_multiplier 민감도

- multiplier가 낮을수록 탐지 민감도 증가 (더 많은 가전에서 PERIODICITY_CHANGE 발생)
- **multiplier=2.0** (현행) 기준 탐지 여부를 확인
- 탐지가 전혀 없으면 multiplier 낮추기 / 탐지가 모든 가전에서 발생하면 높이기

> Periodicity detection은 **LOW severity**만 반환하므로 MEDIUM/HIGH 알림에는 영향 없음.
> 과탐보다는 놓치는 것이 더 나쁜 신호일 수 있어 multiplier=1.5~2.0 권장.

## 7. ANOM-001 vs ANOM-002 보완 비교

같은 평가 구간에서 두 탐지기가 각각 무엇을 잡는지 비교.

In [ ]:
CONSUMPTION_THR = (0.25, 0.40)
RUNTIME_THR     = (0.30, 0.50)
ALWAYS_ON       = {'일반 냉장고', '김치냉장고', '무선공유기/셋톱박스'}

def severity_label(ratio, thr):
    if ratio >= thr[1]: return 'HIGH'
    if ratio >= thr[0]: return 'MEDIUM'
    return None

rows_compare = []
for name in wide_1min.columns:
    on_thr  = ON_THRESHOLDS.get(name, DEFAULT_ON_THR)
    bl      = baseline_1min[name].dropna()
    ev      = eval_1min[name].dropna()
    if bl.empty or ev.empty:
        continue

    # ANOM-001: 소비량 체크
    bl_mean = bl.resample('D').mean().mean()
    ev_mean = ev.mean()
    anom001_cons = None
    if bl_mean > 0:
        r = (ev_mean - bl_mean) / bl_mean
        anom001_cons = severity_label(r, CONSUMPTION_THR)

    # ANOM-001: 런타임 체크
    anom001_rt = None
    if name not in ALWAYS_ON:
        bl_rt = (bl >= on_thr).resample('D').sum().mean()
        ev_rt = (ev >= on_thr).sum()
        if bl_rt > 0:
            r = (ev_rt - bl_rt) / bl_rt
            anom001_rt = severity_label(r, RUNTIME_THR)

    # ANOM-002: Isolation Forest
    model    = models.get(name)
    anom002_if = None
    if model:
        ev_feats   = make_features(ev, on_thr)
        ev_preds   = model.predict(ev_feats)
        ev_ratio   = (ev_preds == -1).mean()
        anom002_if = 'HIGH' if ev_ratio > 0.3 else 'MEDIUM' if ev_ratio > 0.1 else 'LOW' if ev_ratio > 0 else None

    # ANOM-002: Periodicity
    anom002_period = None
    bl_h   = bl.groupby(bl.index.hour).mean()
    ev_h   = ev.groupby(ev.index.hour).mean()
    bl_std = bl.groupby(bl.index.hour).std()
    common = bl_h.index.intersection(ev_h.index)
    if len(common) >= 6:
        diff = (ev_h[common] - bl_h[common]).abs()
        thr  = bl_std[common].mean() * PERIODICITY_STD_MULT
        if diff.max() > thr:
            anom002_period = 'LOW'

    rows_compare.append({
        '가전': name,
        'ANOM001_소비량': anom001_cons or '-',
        'ANOM001_런타임': anom001_rt  or '-',
        'ANOM002_IF':    anom002_if    or '-',
        'ANOM002_주기성': anom002_period or '-',
    })

cmp_df = pd.DataFrame(rows_compare)
print(cmp_df.to_string(index=False))

### 7번 해석 — ANOM-001 vs ANOM-002 보완 관계

| 케이스 | 의미 |
|--------|------|
| ANOM-001만 탐지 | 소비량·런타임 절대값 증가 — 기기 성능 저하 또는 과다 사용 |
| ANOM-002만 탐지 | 패턴 조합 이상 — 사용 시간대 변화, 비정상 동작 |
| 둘 다 탐지 | 복합 이상 — 알림 우선순위 높임 |
| 둘 다 미탐지 | 정상 또는 탐지 범위 외 이상 (감소, 추세 등) |

> ANOM-001은 **증가 방향** 이상에 강하고, ANOM-002는 **패턴 조합·시간대** 이상에 강함.
> 두 탐지기를 `AnomalyService`에서 병렬 실행하여 결과를 합산하는 구조가 적절.

## 8. 종합 소견

In [ ]:
print('=== ANOM-002 검증 결과 요약 ===')
print()
print('[Isolation Forest]')
for name, model in models.items():
    on_thr   = ON_THRESHOLDS.get(name, DEFAULT_ON_THR)
    ev_feats = make_features(eval_1min[name], on_thr)
    ev_ratio = (model.predict(ev_feats) == -1).mean()
    severity = 'HIGH' if ev_ratio > 0.3 else 'MEDIUM' if ev_ratio > 0.1 else 'LOW' if ev_ratio > 0 else 'NORMAL'
    print(f'  {name:10s}: 이상 비율 {ev_ratio*100:.1f}%  [{severity}]')

print()
print('[Periodicity]')
for _, row in sens2_df[sens2_df['std_mult'] == PERIODICITY_STD_MULT].iterrows():
    print(f'  {row["가전"]:10s}: max|편차|={row["max_diff"]}W  임계={row["threshold"]}W  {row["탐지"]}')

print()
print('[체크리스트]')
print('[ ] 3번: 베이스라인 vs 평가 스코어 분포 분리 여부 확인')
print('[ ] 4번: 이상 마커가 실제 비정상 구간에 집중되는지 확인')
print('[ ] 5번: 시간대별 패턴 탐지가 의미 있는 가전에서만 발생하는지 확인')
print('[ ] 6-A: contamination=0.05가 평가 이상 비율 안정적으로 유지하는지 확인')
print('[ ] 6-B: std_multiplier=2.0 기준 탐지 가전 수 적절한지 확인')
print('[ ] 7번: ANOM-001 미탐지 구간을 ANOM-002가 보완하는지 확인')

---

## 최종 알고리즘 결정 — PatternAnomalyDetector v1 (ANOM-002)

### 탐지 전략 개요

```
베이스라인 (4주) → fit(IsolationForest)  →  모델 저장
평가 구간 (24h)  → predict()             →  이상 비율 → severity
                 → hourly profile 비교    →  PERIODICITY_CHANGE (LOW)
```

### 피처 구성

| 피처 | 의미 | 비고 |
|------|------|------|
| `power_w` | 순간 전력 | NILM 출력 |
| `is_on` | ON/OFF 이진값 | 가전별 ON 임계값 적용 |
| `hour` | 시간대 (0~23) | 하루 내 주기성 |
| `dayofweek` | 요일 (0=월) | 주간 주기성 |

### 하이퍼파라미터

| 파라미터 | 기본값 | 역할 | 조정 기준 |
|---------|:------:|------|----------|
| `contamination` | 0.05 | 학습 데이터 내 이상 비율 가정 | 평가 이상 비율이 불안정하면 0.03으로 낮춤 |
| `n_estimators` | 100 | IF 트리 수 | 기본값 유지 (속도·정확도 균형) |
| `periodicity_std_multiplier` | 2.0 | 시간대 편차 허용 배수 | 과탐 시 2.5, 미탐 시 1.5로 조정 |

### Severity 기준 (IF)

| 평가 구간 이상 비율 | Severity |
|:------------------:|:--------:|
| > 30% | HIGH |
| > 10% | MEDIUM |
| > 0% | LOW |
| 0% | — |

### ANOM-001과의 보완 관계

| 탐지 조합 | 권장 대응 |
|-----------|----------|
| ANOM-001만 HIGH | 소비량·런타임 집중 점검 |
| ANOM-002만 MEDIUM+ | 사용 패턴 변화 확인 (시간대, 행동 변화) |
| 둘 다 MEDIUM+ | 복합 이상 — 즉각 점검 권장 |

### 검증 결과 (house_049, 31일)

| 검증 항목 | 결과 |
|-----------|------|
| 피처 분포 | 가전별 시간대·요일 패턴 뚜렷, IF 학습 적합 |
| contamination 민감도 | 0.05 기준 평가 이상 비율 안정적 |
| periodicity 탐지 | std_multiplier=2.0 기준 유의미한 가전에서만 탐지 |
| ANOM-001 보완 | ANOM-001 미탐지(시간대 이상)를 periodicity가 보완 |

### 프로덕션 전환 시 권장 사항

1. 베이스라인 데이터 최소 **2주** 이상 확보 후 fit() 실행
2. 가전 추가 시 `ON_THRESHOLDS` 업데이트 필수
3. 계절 변화 시 모델 재학습 (에어컨 하절기/동절기 분리 고려)
4. `dayofweek` 피처만으로는 공휴일 패턴 미반영 → 추후 `is_holiday` 피처 추가 검토